# Lunar Lander RL Agent Training (DQN)

This notebook trains a reinforcement learning agent to solve the **LunarLander-v3** environment from Gymnasium using **Deep Q-Networks (DQN)**.

**Environment summary**
- **Observation space:** 8-dimensional continuous vector (x, y position; x, y velocity; angle; angular velocity; left/right leg ground contact booleans)
- **Action space:** 4 discrete actions (do nothing, fire left engine, fire main engine, fire right engine)
- **Reward:** shaped reward for moving toward the landing pad, penalties for crashing/fuel use, +100 for safe landing, -100 for crashing
- **Solved threshold:** average reward ≥ 200 over 100 consecutive episodes (the standard benchmark for this environment)

**Algorithm:** DQN with experience replay and a target network — the standard, well-established approach for this environment (Mnih et al., 2015), extended with a few practical improvements (Double DQN target, reward/loss logging, epsilon decay schedule).

> **Note on running this notebook:** This was authored and validated for logical/API correctness, but not executed in the authoring sandbox, since it has no internet access to install `gymnasium`, `box2d`, and `torch`. Run it in Google Colab (free GPU, `!pip install` works out of the box) or any machine with internet access. Full training (≈600–1000 episodes) takes roughly 15–40 minutes on CPU, faster on GPU.


## 1. Install & Import Dependencies

In [ ]:
# Run this cell once. On Google Colab, box2d wheels install without extra system deps.
# If box2d fails to build on your local machine, try: pip install swig  (then re-run this cell)
!pip install -q gymnasium[box2d] torch numpy matplotlib


In [ ]:
import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Explore the Environment

Before writing any agent code, it's worth confirming the observation/action space shapes and looking at a few random-action episodes to sanity check the environment.

In [ ]:
env = gym.make("LunarLander-v3")
obs, info = env.reset(seed=SEED)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Sample observation:", obs)

STATE_DIM = env.observation_space.shape[0]
N_ACTIONS = env.action_space.n
print(f"\nstate_dim={STATE_DIM}, n_actions={N_ACTIONS}")


In [ ]:
# Quick sanity check: run a few episodes with a random policy and report rewards
def run_random_episode(env, seed=None):
    obs, info = env.reset(seed=seed)
    total_reward = 0.0
    steps = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    return total_reward, steps

random_rewards = [run_random_episode(env, seed=i)[0] for i in range(5)]
print("Random-policy episode rewards:", [round(r, 1) for r in random_rewards])
print("(Expect very negative rewards -- a random policy crashes almost every time)")
env.close()


## 3. Q-Network Architecture

A simple fully-connected network mapping the 8-dim state to Q-values for each of the 4 actions. Two hidden layers of 128 units with ReLU activations is a well-established, sufficient architecture for this environment's complexity.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions),
        )

    def forward(self, x):
        return self.net(x)


## 4. Replay Buffer

Experience replay stores transitions `(state, action, reward, next_state, done)` and samples random minibatches for training. This breaks the correlation between consecutive samples and greatly stabilizes training, which is essential for DQN to converge.

In [ ]:
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states = torch.tensor(np.array([t.state for t in batch]), dtype=torch.float32, device=device)
        actions = torch.tensor([t.action for t in batch], dtype=torch.long, device=device).unsqueeze(1)
        rewards = torch.tensor([t.reward for t in batch], dtype=torch.float32, device=device).unsqueeze(1)
        next_states = torch.tensor(np.array([t.next_state for t in batch]), dtype=torch.float32, device=device)
        dones = torch.tensor([t.done for t in batch], dtype=torch.float32, device=device).unsqueeze(1)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


## 5. DQN Agent

Key design choices:
- **Target network**: a lagging copy of the Q-network, updated periodically (or via soft/Polyak update), used to compute stable TD targets. Without this, training is notoriously unstable because the target moves every step.
- **Double DQN**: action *selection* uses the online network, action *evaluation* uses the target network, which reduces the well-known overestimation bias of vanilla DQN.
- **Epsilon-greedy exploration** with exponential decay from `eps_start` to `eps_end`.
- **Huber loss (smooth L1)** instead of MSE, which is more robust to outlier TD errors.

In [ ]:
class DQNAgent:
    def __init__(self, state_dim, n_actions, lr=5e-4, gamma=0.99,
                 buffer_capacity=100_000, batch_size=64,
                 target_update_every=1000, tau=None):
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_every = target_update_every
        self.tau = tau  # if set, use soft (Polyak) target updates instead of hard periodic copy
        self.learn_step = 0

        self.q_net = QNetwork(state_dim, n_actions).to(device)
        self.target_net = QNetwork(state_dim, n_actions).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_capacity)

    def act(self, state, epsilon):
        if random.random() < epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q_values = self.q_net(state_t)
            return int(q_values.argmax(dim=1).item())

    def store(self, *transition_args):
        self.buffer.push(*transition_args)

    def update_target_network(self):
        if self.tau is None:
            self.target_net.load_state_dict(self.q_net.state_dict())
        else:
            for target_param, param in zip(self.target_net.parameters(), self.q_net.parameters()):
                target_param.data.copy_(self.tau * param.data + (1.0 - self.tau) * target_param.data)

    def learn(self):
        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        # Current Q estimates for the actions actually taken
        q_values = self.q_net(states).gather(1, actions)

        with torch.no_grad():
            # Double DQN: select best next action with online network, evaluate with target network
            next_actions = self.q_net(next_states).argmax(dim=1, keepdim=True)
            next_q_values = self.target_net(next_states).gather(1, next_actions)
            targets = rewards + self.gamma * next_q_values * (1 - dones)

        loss = F.smooth_l1_loss(q_values, targets)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        self.learn_step += 1
        if self.tau is not None:
            self.update_target_network()  # soft update every learning step
        elif self.learn_step % self.target_update_every == 0:
            self.update_target_network()  # hard update periodically

        return loss.item()


## 6. Training Loop

Epsilon decays exponentially from `eps_start` to `eps_end` over `eps_decay` steps, so the agent explores heavily early on and exploits its learned policy later. We track per-episode reward and the 100-episode moving average (the environment's official "solved" metric).

In [ ]:
def train_dqn(
    n_episodes=1000,
    max_steps=1000,
    gamma=0.99,
    lr=5e-4,
    batch_size=64,
    buffer_capacity=100_000,
    eps_start=1.0,
    eps_end=0.01,
    eps_decay=0.995,      # multiplicative decay applied once per episode
    target_update_every=1000,
    solved_threshold=200.0,
    solved_window=100,
    log_every=20,
):
    env = gym.make("LunarLander-v3")
    agent = DQNAgent(
        state_dim=env.observation_space.shape[0],
        n_actions=env.action_space.n,
        lr=lr, gamma=gamma,
        buffer_capacity=buffer_capacity, batch_size=batch_size,
        target_update_every=target_update_every,
    )

    epsilon = eps_start
    episode_rewards = []
    losses = []

    for episode in range(1, n_episodes + 1):
        state, info = env.reset(seed=SEED + episode)
        episode_reward = 0.0
        episode_losses = []

        for step in range(max_steps):
            action = agent.act(state, epsilon)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            agent.store(state, action, reward, next_state, float(done))
            loss = agent.learn()
            if loss is not None:
                episode_losses.append(loss)

            state = next_state
            episode_reward += reward

            if done:
                break

        epsilon = max(eps_end, epsilon * eps_decay)
        episode_rewards.append(episode_reward)
        if episode_losses:
            losses.append(np.mean(episode_losses))

        if episode % log_every == 0:
            avg_last_100 = np.mean(episode_rewards[-solved_window:])
            print(f"Episode {episode:4d} | Reward: {episode_reward:7.1f} | "
                  f"Avg(last {min(len(episode_rewards), solved_window)}): {avg_last_100:7.1f} | "
                  f"Epsilon: {epsilon:.3f}")

        # Early stopping once solved
        if len(episode_rewards) >= solved_window:
            avg_last_100 = np.mean(episode_rewards[-solved_window:])
            if avg_last_100 >= solved_threshold:
                print(f"\nSolved in {episode} episodes! "
                      f"Average reward over last {solved_window} episodes: {avg_last_100:.1f}")
                break

    env.close()
    return agent, episode_rewards, losses


## 7. Run Training

This is the main training cell. Expect ~400-800 episodes to reach the solved threshold with these hyperparameters, though this varies with random seed. Adjust `n_episodes` down for a quicker smoke test, or up if it hasn't converged.

In [ ]:
agent, episode_rewards, losses = train_dqn(
    n_episodes=1000,
    gamma=0.99,
    lr=5e-4,
    batch_size=64,
    eps_start=1.0,
    eps_end=0.01,
    eps_decay=0.995,
    target_update_every=1000,
    solved_threshold=200.0,
    log_every=20,
)


## 8. Plot Learning Curves

In [ ]:
def moving_average(x, window=100):
    if len(x) < window:
        return np.array([])
    return np.convolve(x, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episode_rewards, alpha=0.35, label="Episode reward")
ma = moving_average(episode_rewards, 100)
if len(ma) > 0:
    axes[0].plot(range(99, 99 + len(ma)), ma, color="darkred", linewidth=2, label="100-episode moving avg")
axes[0].axhline(200, color="green", linestyle="--", label="Solved threshold (200)")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward")
axes[0].set_title("Training Reward per Episode")
axes[0].legend()

if losses:
    axes[1].plot(losses, color="steelblue")
    axes[1].set_xlabel("Episode")
    axes[1].set_ylabel("Avg Huber Loss")
    axes[1].set_title("Training Loss per Episode")

plt.tight_layout()
plt.show()


## 9. Save the Trained Model

In [ ]:
torch.save(agent.q_net.state_dict(), "lunar_lander_dqn.pt")
print("Model saved to lunar_lander_dqn.pt")


## 10. Evaluate the Trained Agent

Run the trained policy greedily (epsilon=0, no exploration) for a number of episodes and report statistics. This is the number that matters for judging whether the agent actually solved the task, since training reward includes exploration noise.

In [ ]:
def evaluate_agent(agent, n_episodes=100, render=False):
    env = gym.make("LunarLander-v3", render_mode="human" if render else None)
    rewards = []
    for ep in range(n_episodes):
        state, info = env.reset(seed=1000 + ep)
        total_reward = 0.0
        done = False
        while not done:
            action = agent.act(state, epsilon=0.0)  # fully greedy, no exploration
            state, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)
    env.close()
    return rewards

eval_rewards = evaluate_agent(agent, n_episodes=100, render=False)
print(f"Evaluation over {len(eval_rewards)} episodes (greedy policy, epsilon=0):")
print(f"  Mean reward:   {np.mean(eval_rewards):.1f}")
print(f"  Std reward:    {np.std(eval_rewards):.1f}")
print(f"  Min / Max:     {np.min(eval_rewards):.1f} / {np.max(eval_rewards):.1f}")
print(f"  Success rate (reward >= 200): {np.mean(np.array(eval_rewards) >= 200) * 100:.1f}%")

plt.figure(figsize=(8, 4.5))
plt.hist(eval_rewards, bins=20, color="mediumseagreen", edgecolor="black")
plt.axvline(200, color="red", linestyle="--", label="Solved threshold")
plt.xlabel("Episode Reward")
plt.ylabel("Count")
plt.title("Evaluation Reward Distribution (Greedy Policy)")
plt.legend()
plt.tight_layout()
plt.show()


## 11. (Optional) Watch the Agent Land

Set `render=True` to open a window and watch the trained agent land in real time. This requires a display — it won't work in a headless environment (like a remote server or most cloud notebooks without a virtual display), but works fine locally or with Colab's `render_mode="rgb_array"` + video capture (see cell below for that alternative).

In [ ]:
# Uncomment to watch with a live window (local machine only):
# _ = evaluate_agent(agent, n_episodes=3, render=True)

# Colab-friendly alternative: capture frames and save as a video/gif instead of a live window
import imageio

def record_episode(agent, filename="lunar_lander_demo.gif", seed=0):
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    state, info = env.reset(seed=seed)
    frames = [env.render()]
    done = False
    total_reward = 0.0
    while not done:
        action = agent.act(state, epsilon=0.0)
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
        frames.append(env.render())
    env.close()
    imageio.mimsave(filename, frames, fps=30)
    print(f"Saved {len(frames)} frames to {filename} (episode reward: {total_reward:.1f})")

# record_episode(agent, "lunar_lander_demo.gif")  # uncomment to run (requires: pip install imageio)


## 12. Summary & Discussion

**Approach:** Double DQN with experience replay and a target network, trained on `LunarLander-v3` from Gymnasium.

**Key hyperparameters used:**
- Learning rate: 5e-4 (Adam optimizer)
- Discount factor (gamma): 0.99
- Replay buffer capacity: 100,000 transitions
- Batch size: 64
- Epsilon decay: 1.0 → 0.01, multiplicative 0.995 per episode
- Target network: hard update every 1000 learning steps
- Network: 2 hidden layers of 128 units, ReLU activations

**What to look for in your results:**
- The reward curve should be very noisy and negative for the first 50-150 episodes (mostly exploration/crashing), then climb steadily as epsilon decays and the Q-network learns useful value estimates.
- The 100-episode moving average crossing 200 is the standard "solved" criterion for this environment.
- If training is unstable (reward oscillates wildly or collapses after initially improving) — common failure modes are: learning rate too high, target network updated too frequently, or replay buffer too small relative to batch size early in training.

**Possible extensions:**
- **Dueling DQN**: separate value and advantage streams, often improves sample efficiency.
- **Prioritized Experience Replay**: sample transitions with high TD-error more often instead of uniformly.
- **PPO or A2C** (policy-gradient/actor-critic): often more stable than DQN on this environment and worth comparing against, especially via `stable-baselines3` (`pip install stable-baselines3`) for a well-tested reference implementation.
- **Hyperparameter sweep**: try different `eps_decay` schedules, hidden sizes, or learning rates and compare convergence speed.
- **Continuous action variant**: `LunarLanderContinuous-v3` requires a different algorithm (e.g., DDPG, TD3, or SAC) since DQN only handles discrete actions.
